# Fraud Detection – Logistic Regression vs XGBoost vs LightGBM

## Load & explore

In [62]:
import pandas as pd
import numpy as np

In [63]:
df = pd.read_csv("../../Data/Raw/fraudTrain.csv")
df = df.drop(columns=["Unnamed: 0"])
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
df.head()

,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [64]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 22 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   trans_date_trans_time  1296675 non-null  datetime64[us]
 1   cc_num                 1296675 non-null  int64         
 2   merchant               1296675 non-null  str           
 3   category               1296675 non-null  str           
 4   amt                    1296675 non-null  float64       
 5   first                  1296675 non-null  str           
 6   last                   1296675 non-null  str           
 7   gender                 1296675 non-null  str           
 8   street                 1296675 non-null  str           
 9   city                   1296675 non-null  str           
 10  state                  1296675 non-null  str           
 11  zip                    1296675 non-null  int64         
 12  lat                    1296675 non-null

In [65]:
df.isna().sum()

trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
dtype: int64

In [66]:
print(df["is_fraud"].value_counts())
print(df["is_fraud"].value_counts(normalize=True) * 100)

is_fraud
0    1289169
1       7506
Name: count, dtype: int64
is_fraud
0    99.421135
1     0.578865
Name: proportion, dtype: float64


In [67]:
df.groupby("is_fraud")["amt"].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,1289169.0,67.667110,154.007971,1.00,9.6100,47.280,82.540,28948.90
1,7506.0,531.320092,390.560070,1.06,245.6625,396.505,900.875,1376.04


In [68]:
df["hour"] = df["trans_date_trans_time"].dt.hour
df.groupby("hour")["is_fraud"].mean()

hour
0     0.014940
1     0.015349
2     0.014652
3     0.014239
4     0.001099
5     0.001423
6     0.000946
7     0.001327
8     0.001153
9     0.001114
10    0.000946
11    0.000998
12    0.001027
13    0.001225
14    0.001325
15    0.001208
16    0.001156
17    0.001192
18    0.001226
19    0.001236
20    0.000952
21    0.001129
22    0.028829
23    0.028374
Name: is_fraud, dtype: float64

In [69]:
df.groupby("category")["is_fraud"].agg(["count", "mean"]).sort_values("mean", ascending=False)

,count,mean
category,,
shopping_net,97543,0.017561
misc_net,63287,0.014458
grocery_pos,123638,0.014098
shopping_pos,116672,0.007225
gas_transport,131659,0.004694
misc_pos,79655,0.003139
grocery_net,45452,0.002948
travel,40507,0.002864
entertainment,94014,0.002478


## Feature engineering

In [70]:
# drop identifiers / free text (cc_num and dob are used below, so they stay for now)
df = df.drop(columns=["street", "city", "first", "last", "job", "trans_num", "unix_time", "zip"])

In [71]:
# time + age features
transaction_time = df["trans_date_trans_time"]
day_of_week = transaction_time.dt.dayofweek
df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 3)).astype(int)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["day_sin"] = np.sin(2 * np.pi * day_of_week / 7)
df["day_cos"] = np.cos(2 * np.pi * day_of_week / 7)
dob = pd.to_datetime(df["dob"], errors="coerce")
df["age"] = (transaction_time - dob).dt.days / 365.25

In [72]:
# distance between customer and merchant (haversine, km)
lat1, lat2 = np.radians(df["lat"]), np.radians(df["merch_lat"])
dlat = lat2 - lat1
dlon = np.radians(df["merch_long"] - df["long"])
a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
df["distance_km"] = 6371 * 2 * np.arcsin(np.sqrt(a.clip(0, 1)))

In [73]:
# per-customer history (uses only PAST transactions)
df = df.sort_values(["cc_num", "trans_date_trans_time"]).reset_index(drop=True)

prior_median = df.groupby("cc_num")["amt"].transform(
    lambda amounts: amounts.shift().expanding(min_periods=1).median()
)
df["customer_prior_tx_count"] = df.groupby("cc_num").cumcount()

# log amount difference; large positive = unusually large payment
df["amount_vs_customer_median"] = (np.log1p(df["amt"]) - np.log1p(prior_median)).fillna(0)

In [74]:
# transactions in the last 1h / 24h (excluding the current one)
rolling_source = df.assign(_transaction=1).set_index("trans_date_trans_time")

for window, feature in [("1h", "tx_count_1h"), ("24h", "tx_count_24h")]:
    counts = (
        rolling_source.groupby("cc_num")["_transaction"]
        .rolling(window, closed="left")
        .sum()
        .reset_index(level=0, drop=True)
    )
    df[feature] = counts.fillna(0).to_numpy(dtype="int32")   # fillna BEFORE the int cast

In [75]:
df.head(1)

,trans_date_trans_time,cc_num,merchant,category,amt,gender,state,lat,long,city_pop,...,hour_sin,hour_cos,day_sin,day_cos,age,distance_km,customer_prior_tx_count,amount_vs_customer_median,tx_count_1h,tx_count_24h
0,2019-01-01 12:47:15,60416207185,"fraud_Jones, Sawayn and Romaguera",misc_net,7.27,F,WY,43.0048,-108.8964,1645,...,1.224647e-16,-1.0,0.781831,0.62349,32.870637,127.606239,0,0.0,0,0


## Train / test split

In [76]:
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [77]:
cat_cols = ["category", "gender", "state"]
merchant_col = ["merchant"]
num_cols = [
    "amt", "lat", "long", "city_pop", "merch_lat", "merch_long",
    "is_night", "hour_sin", "hour_cos", "day_sin", "day_cos",
    "age", "distance_km",
    "customer_prior_tx_count", "amount_vs_customer_median",
    "tx_count_1h", "tx_count_24h",
]
feature_cols = cat_cols + merchant_col + num_cols

In [78]:
# Chronological 80/20 split: earlier transactions train, latest transactions validate
_df_time_sorted = df.sort_values("trans_date_trans_time").reset_index(drop=True)
split_idx = int(len(_df_time_sorted) * 0.8)
train_df = _df_time_sorted.iloc[:split_idx]
valid_df = _df_time_sorted.iloc[split_idx:]

x_train = train_df[feature_cols]
y_train = train_df["is_fraud"]
x_test = valid_df[feature_cols]
y_test = valid_df["is_fraud"]

print("Train period:", train_df["trans_date_trans_time"].min(), "to", train_df["trans_date_trans_time"].max())
print("Validation period:", valid_df["trans_date_trans_time"].min(), "to", valid_df["trans_date_trans_time"].max())
print("Train fraud rate:", y_train.mean())
print("Validation fraud rate:", y_test.mean())

Train period: 2019-01-01 00:00:18 to 2020-03-06 07:15:17
Validation period: 2020-03-06 07:16:43 to 2020-06-21 12:13:37
Train fraud rate: 0.005753176393467908
Validation fraud rate: 0.005930553145545337


## Preprocessing (shared by all models)

In [79]:
preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("merchant_target", TargetEncoder(target_type="binary", smooth="auto", cv=5, random_state=42), merchant_col),
    ("numeric", StandardScaler(), num_cols),
])

# fraud is rare, so weight the positive class for the boosted models
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

## Evaluation helper

In [80]:
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

results = {}

def evaluate(name, model):
    y_pred = model.predict(x_test)
    y_score = model.predict_proba(x_test)[:, 1]

    print(classification_report(
        y_test, y_pred, labels=[0, 1],
        target_names=["Legitimate", "Fraud"], zero_division=0
    ))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred, labels=[0, 1]))
    ap = average_precision_score(y_test, y_score)
    print("Average precision:", ap)
    results[name] = ap

print("Fraud rate baseline:", y_test.mean())

Fraud rate baseline: 0.005930553145545337


## 1. Logistic Regression

In [81]:
log_reg = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("LogisticRegression", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
log_reg.fit(x_train, y_train)
evaluate("Logistic Regression", log_reg)

c:\Users\arije\anaconda3\envs\ai_env\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


              precision    recall  f1-score   support

  Legitimate       1.00      0.97      0.98    257797
       Fraud       0.13      0.85      0.23      1538

    accuracy                           0.97    259335
   macro avg       0.57      0.91      0.61    259335
weighted avg       0.99      0.97      0.98    259335

Confusion matrix:
 [[249277   8520]
 [   225   1313]]
Average precision: 0.30398211499774197


## 2. XGBoost

In [82]:
xgb = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("XGBoost", XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        eval_metric="aucpr", tree_method="hist",
        n_jobs=-1, random_state=42,
    )),
])
xgb.fit(x_train, y_train)
evaluate("XGBoost", xgb)

c:\Users\arije\anaconda3\envs\ai_env\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00    257797
       Fraud       0.69      0.87      0.77      1538

    accuracy                           1.00    259335
   macro avg       0.84      0.93      0.88    259335
weighted avg       1.00      1.00      1.00    259335

Confusion matrix:
 [[257192    605]
 [   201   1337]]
Average precision: 0.8866002694725328


## 3. LightGBM

In [83]:
lgbm = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("LightGBM", LGBMClassifier(
        n_estimators=300, num_leaves=31, learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1, random_state=42, verbose=-1,
    )),
])
lgbm.fit(x_train, y_train)
evaluate("LightGBM", lgbm)

c:\Users\arije\anaconda3\envs\ai_env\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


              precision    recall  f1-score   support

  Legitimate       1.00      0.96      0.98    257797
       Fraud       0.12      0.90      0.21      1538

    accuracy                           0.96    259335
   macro avg       0.56      0.93      0.59    259335
weighted avg       0.99      0.96      0.97    259335

Confusion matrix:
 [[247289  10508]
 [   152   1386]]
Average precision: 0.11993152042425731


## Comparison

In [84]:
pd.Series(results, name="Average precision").sort_values(ascending=False).to_frame()

,Average precision
XGBoost,0.886600
Logistic Regression,0.303982
LightGBM,0.119932


## Final test on fraudTest.csv

In [85]:
def make_features(d):
    """Create the same engineered features used for model training."""
    d = d.drop(columns=["Unnamed: 0"], errors="ignore")
    d["trans_date_trans_time"] = pd.to_datetime(d["trans_date_trans_time"])
    d["hour"] = d["trans_date_trans_time"].dt.hour
    d = d.drop(columns=["street", "city", "first", "last", "job", "trans_num", "unix_time", "zip"])

    day_of_week = d["trans_date_trans_time"].dt.dayofweek
    d["is_night"] = ((d["hour"] >= 22) | (d["hour"] <= 3)).astype(int)
    d["hour_sin"] = np.sin(2 * np.pi * d["hour"] / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d["hour"] / 24)
    d["day_sin"] = np.sin(2 * np.pi * day_of_week / 7)
    d["day_cos"] = np.cos(2 * np.pi * day_of_week / 7)
    dob = pd.to_datetime(d["dob"], errors="coerce")
    d["age"] = (d["trans_date_trans_time"] - dob).dt.days / 365.25

    lat1, lat2 = np.radians(d["lat"]), np.radians(d["merch_lat"])
    dlat = lat2 - lat1
    dlon = np.radians(d["merch_long"] - d["long"])
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    d["distance_km"] = 6371 * 2 * np.arcsin(np.sqrt(a.clip(0, 1)))

    d = d.sort_values(["cc_num", "trans_date_trans_time"]).reset_index(drop=True)
    prior_median = d.groupby("cc_num")["amt"].transform(
        lambda amounts: amounts.shift().expanding(min_periods=1).median()
    )
    d["customer_prior_tx_count"] = d.groupby("cc_num").cumcount()
    d["amount_vs_customer_median"] = (np.log1p(d["amt"]) - np.log1p(prior_median)).fillna(0)

    src = d.assign(_transaction=1).set_index("trans_date_trans_time")
    for window, feature in [("1h", "tx_count_1h"), ("24h", "tx_count_24h")]:
        counts = (
            src.groupby("cc_num")["_transaction"]
            .rolling(window, closed="left").sum()
            .reset_index(level=0, drop=True)
        )
        d[feature] = counts.fillna(0).to_numpy(dtype="int32")
    return d

In [86]:
test_df = make_features(pd.read_csv("../../Data/Raw/fraudTest.csv"))
x_final = test_df[feature_cols]
y_final = test_df["is_fraud"]
print("fraudTest fraud rate:", y_final.mean())

fraudTest fraud rate: 0.0038598644278853163


In [87]:
final_score = xgb.predict_proba(x_final)[:, 1]

print("Average precision:", average_precision_score(y_final, final_score))


Average precision: 0.8597342655040864


In [88]:
from pathlib import Path
import joblib

models = {
    "Logistic Regression": log_reg,
    "XGBoost": xgb,
    "LightGBM": lgbm,
}

best_name = max(results, key=results.get)
best_model = models[best_name]

model_path = Path("../../Models/best_fraud_model.joblib")
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, model_path)

print(f"Saved {best_name} (average precision: {results[best_name]:.4f}) to {model_path}")

Saved XGBoost (average precision: 0.8866) to ..\..\Models\best_fraud_model.joblib
